# Proximal Policy Optimization

$\theta$: parameters of the policy network; and the policy can be defined as: $\pi_{\theta}(a|s)$, which equals the probability of taking action $a$ in state $s$.
1. For discrete actions: the network gives logits $\longrightarrow$ softmax $\longrightarrow$ probabilities
2. For continous actions: network gives: mean and log($\sigma$) of the Gaussian distrbution of (action probabilities)

Let's have the cost function as the expected return under policy $\pi(\theta)$:

$$ J(\theta) = \mathbb{E}_{\pi_{\theta}}(\mathcal{G}_t) = \mathbb{E}_{\pi_{\theta}}(R_t + \gamma \mathcal{G}_{t+1}) $$

Now, we know return $\mathcal{G}(t)$ depends on which states the MDP visits, which is directly dictated by the underlying system dynamics as well as the policy $\pi$. Let $\tau$ be such a trajectory, hence we can write the following:

$$ J(\theta) = \sum_{\forall \tau} p(\tau | \theta) \mathcal{G}(\tau) $$

$\tau$ is the set of trajectories and any one trajectory can be defined as: $(s_0, a_0, r_0, s_1, a_1, r_1, ..., s_n, a_n, r_n)$ followed when using policy $\pi_{\theta}$. And $p(\tau | \theta)$ is the probability to follow trajectory $\tau$ under policy $\pi_{\theta}$.

# Probability of Following a trajectory $\tau$

$$ P(\tau | \theta) = p(s_0) \prod^{T-1}_{t=0} \pi_{\theta} (a_t | s_t) \times p(s_{t+1} | s_t, a_t) $$

Since $J(\theta)$ is the expected value of return, we want to maximise $J$. We use gradient descent with respect to $\theta$ because we need to find:

$$ \theta': \theta' = \text{argmax}_{\theta}J(\theta) $$

Taking gradient of $J(\theta)$ we get, 

$$ \nabla_{\theta} J(\theta) = \nabla_{\theta} \mathbb{E}_{\pi_{\theta}} \mathcal{G}(\tau) $$

$$ \nabla_{\theta} J(\theta) = \nabla_{\theta} \sum_{\forall \tau} p(\tau | \theta) \mathcal{G}(\tau) $$

Converting summation to integral by considering infinitisimal variations among $\tau$:

$$ \nabla_{\theta} J(\theta) = \nabla_{\theta} \int_{\tau} p(\tau | \theta) \mathcal{G}(\tau) d\tau$$

Using the $\nabla_{\theta} p = p \nabla_{\theta} log(p)$ trick, and taking the $\nabla$ inside the integral we get,

$$ \nabla_{\theta} J(\theta) = \int_{\tau} \nabla_{\theta} p(\tau | \theta) \mathcal{G}(\tau) d\tau$$

$$ \nabla_{\theta} J(\theta) = \int_{\tau} p(\tau | \theta) \nabla_{\theta} log (p(\tau | \theta)) \mathcal{G}(\tau) d\tau $$

Now, let's solve for $\nabla_{\theta} log (p(\tau | \theta))$:

$$ \nabla_{\theta} log (p(\tau | \theta)) = \nabla_{\theta} log \left\{ p(s_0) \prod^{T-1}_{t=0} \pi_{\theta} (a_t | s_t) \times p(s_{t+1} | s_t, a_t)) \right\} $$

$$ \nabla_{\theta} log (p(\tau | \theta)) = \nabla_{\theta} \left\{ log p(s_0) + \sum_{t=0}^{T-1} \left\{ log(\pi_{\theta} (a_t | s_t)) + log(p(s_{t+1} | s_t, a_t)) \right\} \right \}$$

The terms which don't depend on $\theta$ their derivative is going to be zero, hence,

$$ \nabla_{\theta} log (p(\tau | \theta)) = \nabla_{\theta} \sum_{t=0}^{T-1} log(\pi_{\theta} (a_t | s_t)) $$

So, what does this say? Take the gradient of the sum of log of all action probabilities at all times for that particular trajectory. Now, substitute this back in the integral we found earlier: 

$$ \nabla_{\theta} J(\theta) = \int_{\tau} p(\tau | \theta) \nabla_{\theta} log (p(\tau | \theta)) \mathcal{G}(\tau) d\tau $$

This structure when morphed back to the Expected value form looks like:

$$ \nabla_{\theta} J(\theta) = \mathbb{E}_{\pi_{\theta}} \left \{ \nabla_{\theta} log (p(\tau | \theta)) \mathcal{G}(\tau) \right \} $$

$$ \nabla_{\theta} J(\theta) = \mathbb{E}_{\tau \sim \pi_{\theta}} \left \{ \left\{ \sum_{t=0}^{T-1} \nabla_{\theta} log(\pi_{\theta} (a_t | s_t)) \right \} \mathcal{G}(\tau) \right \} $$

## Replacing $G(\tau)$ with $Q^{\pi}(s, a)$

Look at the expression above, it says, take the expected value over all the set of trajectories, which are reachable through when following a policy $\pi_{\theta}$. Let's consider the trajectories to be $\tau:= \tau_1, \tau_2, \tau_3, \dots, \tau_n $ and so on.

$$ \tau_1 := \{s_{(0, 1)}, a_{(0, 1)}, r_{(0, 1)}, s_{(1, 1)}, a_{(1, 1)}, r_{(1, 1)}, s_{(2, 1)}, a_{(2, 1)}, r_{(2, 1)}, \dots, s_{(n, 1)}, a_{(n, 1)}, r_{(n, 1)} \} $$

Now, what is the probability of encountering this particular trajectory, when following the policy $\pi_{\theta}$:

 $$ p(\tau = \tau_1 | \theta) = p(s_0) \prod^{T-1}_{t=0} \pi_{\theta}(a_t | s_t) p(s_{t+1} | s_t, a_t) $$

Coming back to, $\nabla_{\theta} J(\theta)$,

$$\nabla_{\theta} J(\theta) = 
 \sum_{\forall \tau} p(\tau | \theta) \sum^{T}_{t=0}\nabla log \pi_{\theta} (a_t | s_t) G(t) $$ 

Swapping the sums (possible due to linearity),

$$ \nabla_{\theta} J(\theta) = 
\sum^{T \longrightarrow \infty}_{t=0} \sum_{\forall \tau} p(\tau | \theta) \nabla log \pi_{\theta} (a_t | s_t) G(t) $$ 

$$ \nabla_{\theta} J(\theta) = 
\sum^{T}_{t=0} \sum_{\forall \tau} p(\tau | \theta) \nabla log \pi_{\theta} (a_t | s_t) G(t) \times \mathbb{1}_{t \leq T} $$ 

The end term $\mathbb{1}_{t \leq T}$ is added to make sure the sum is finite, because in usual cases: $T \longrightarrow \infty $. Now, the inner sum is a sum over all the trajectories encountered when following policy $\pi_{\theta}$. 

What can we conclude here?

$$\sum_{\tau} p(\tau | \theta) = \sum_{\forall s, a} \sum_{\tau: s_t = s, a_t = a} p(\tau | \theta) $$

Substitute it back in the earlier equations we have found out, 

$$ \nabla_{\theta} J(\theta) = 
\sum^{T}_{t=0} \sum_{\forall s, a} \sum_{\tau: s_t = s, a_t = a} p(\tau | \theta) \nabla log \pi_{\theta} (a_t | s_t) \mathbb{E} \left \{ G(t) | s_t = s, a_t = a\right \} \times \mathbb{1}_{t \leq T} $$ 

Now, we know, that, 

$$ \mathbb{E} \left \{ G(t) | s_t = s, a_t = a\right \} = Q^{\pi}(s, a) $$

hence, substituting it back we get, 

$$ \nabla_{\theta} J(\theta) = 
\sum^{T}_{t=0} \sum_{\forall s, a} \sum_{\tau: s_t = s, a_t = a} p(\tau | \theta) \nabla log \pi_{\theta} (a_t | s_t) Q^{\pi}(s, a) \times \mathbb{1}_{t \leq T} $$ 




# On-Policy Methods

1. Policy determines action probabilities. 
2. Those actions, combined with environment dynamics determine which states are visited. 
3. Policy gradient estimates are then weighted by the visited state-action pairs. 
4. In On-Policy methods, the state-action pair transition data comes from the policy whose gradient we are trying to estimate. 

## What are Value Baselines?

We have,

$$ \nabla_{\theta} J(\theta) = \mathbb{E}\left[ \nabla_{\theta} log \pi_{\theta} (a_k | s_k) G_t \right] $$

$G_t$ can be very noisy: environment statistics, different episode lengths, and so on and so forth. Let's normalize it, and subtract a baseline. 

$$ \nabla_{\theta} J(\theta) = 
\mathbb{E} \left[ \nabla_{\theta} log \pi_{\theta} (a_k | s_k) \left\{ G_t - b(s_t) \right \} \right] $$

What is this baseline $b(s_t)$? It's the average expected return from this state, irrespective of what action is chosen. So, try guessing what's the best candidate to substitute this? Guessed it right, it's the State value function: $V^{\pi}(s_t)$!

$$ \nabla_{\theta} J(\theta) = 
\mathbb{E} \left[ \nabla_{\theta} log \pi_{\theta} (a_k | s_k) \left\{ G_t - V^{\pi}(s_t) \right \} \right] $$

## The Baseline Effect

Does subtracting a baseline have any effect on $\nabla_{\theta} J(\theta)$ at all?

$$ \nabla_{\theta} J(\theta) = \mathbb{E} \left[ \nabla_{\theta} log \pi_{\theta} (a_k | s_k) \mathcal{G}_t - \nabla_{\theta} log \pi_{\theta} (a_k | s_k) V^{\pi}(s) \right] $$

Let's expand the second term:

$$ \mathbb{E} \left[ \nabla_{\theta} log \pi_{\theta} (a_k | s_k) V^{\pi}(s) \right] $$

Since $V^{\pi}(s)$ doesn't depend on $\theta$, 

$$ V^{\pi}(s) \mathbb{E} \left[ \nabla_{\theta} log \pi_{\theta} (a_k | s_k) \right] $$

$$ V^{\pi}(s) \mathbb{E} \left[ \frac{\nabla_{\theta} \pi_{\theta} (a_k | s_k)}{\pi_{\theta} (a_k | s_k)} \right] $$

Expand the expected value expression, 

$$ V^{\pi}(s) \sum_{\forall a} \pi(a|s) \frac{\nabla_{\theta} \pi_{\theta} (a_k | s_k)}{\pi_{\theta} (a_k | s_k)} $$

$$ V^{\pi}(s) \nabla_{\theta} \sum_{\forall a} \pi_{\theta} (a_k | s_k) $$

$$ V^{\pi}(s) \nabla_{\theta} 1 = 0 $$

Hence, provde that there is no baseline effect. 

# Advantage Function

$$ A^{\pi}(s, a) = Q^{\pi}(s, a) - V^{\pi}(s) $$

1. What is $Q^{\pi}(s, a)$?

$$ Q^{\pi}(s, a) = \mathbb{E}_{\pi}(Q^{\pi}(s, a)) $$

expected value of return, while being in state $s$, taking action $a$, and then following policy $\pi$ onwards from there. 

2. What is $V^{\pi}(s)$?

$$ V^{\pi}(s) = \sum_{\forall a} \pi(a|s) Q^{\pi}(s, a)$$

Multiplying the expected return from taking a particular action $a$, with the probability of taking that action and then summing it over all possible actiosn in the action space, gives us the state value function of that state.

Hence, the advantage function: $  A^{\pi}(s, a) = Q^{\pi}(s, a) - V^{\pi}(s) $ is just telling us how much better/wose was the action $a$ as compared to the policy's average in state $s$.

# Actor-Critic

In the Advantage function, the $V^{\pi}(s)$ acts are the judgemnt factor, based on the 
$Q^{\pi}(s, a) - V^{\pi}(s)$, we can judge if $a$ was good/equal or worse than the average. Since $V^{\pi}(s)$ provides feedback, it's called the critic. 

Since we don't know $V^{\pi}(s)$ exactly, let's approximate ith with $V^{\phi}(s)$. 

Train the critic's parameters on predicted future return: $L_v(\phi) = (V^{\phi}(s_t) - G_t)^2$

Actor uses:

$$ \nabla_{\theta} J(\theta) =
 \sum^{T-1}_{t=0} \nabla_{\theta} log \pi_{\theta}(a_k|s_k) \hat{A}_t $$

 $$ \hat{A}_t = G_t - V^{\phi}(t) $$

 The critic doesn't directly choose the actions, it evaluates how favourable the observed outcome was, giving the actor a lower variance signal to act on. 